# Feature Engineering

In this phase, raw variables are transformed into meaningful features to improve the performance of machine learning models. The goal is to capture behavioral, financial, and operational patterns that influence product returns.


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\krish\Downloads\ecommerce_returns_clean.csv")  # or your processed dataset from EDA phase
df.head()

,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivered_date,...,returned,request_date,return_reason,total_amount,shipping_cost,profit_margin,customer_age,customer_gender,delivery_days,age_group
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,2023-12-23,2023-12-27,...,0,NaN,NaN,139.47,7.88,31.17,60,Female,4,46-60
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,2025-04-03,2025-04-09,...,0,NaN,NaN,24.73,4.60,-2.62,37,Male,6,36-45
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,2024-10-08,2024-10-12,...,0,NaN,NaN,166.80,6.58,13.44,34,Male,4,26-35
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,2024-09-14,2024-09-20,...,0,NaN,NaN,63.67,5.50,2.14,21,Female,6,18-25
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,2024-12-21,2024-12-27,...,0,NaN,NaN,13.88,2.74,1.15,39,Male,6,36-45


## 4.1 Category Return Risk (Target Encoding)

This feature captures the historical return tendency of each product category.

In [ ]:
category_return_risk = df.groupby("category")["returned"].mean()
df["category_return_risk"] = df["category"].map(category_return_risk)

## 4.2 Region Return Risk

This feature captures return behavior differences across regions.

In [4]:
region_return_risk = df.groupby("region")["returned"].mean()
df["region_return_risk"] = df["region"].map(region_return_risk)

## 4.3 Delivery Speed Grouping

Delivery days are converted into meaningful customer experience categories.

In [5]:
df["delivery_speed"] = pd.cut(
    df["delivery_days"],
    bins=[0, 3, 7, df["delivery_days"].max()],
    labels=["Fast", "Standard", "Slow"]
)

## 4.4 Discount Band Creation

Discount values are grouped to capture non-linear effects on returns.

In [7]:
df["discount_band"] = pd.cut(
    df["discount"],
    bins=[-0.01, 0.0, 0.1, 0.2, 0.3],
    labels=["No Discount", "Low", "Medium", "High"]
)

## 4.5 High Value Order Flag

Identifies high-value transactions that may have different return behavior.

In [8]:
df["high_value_order"] = (df["total_amount"] > df["total_amount"].median()).astype(int)

## 4.6 Price × Discount Interaction

Captures combined effect of pricing and discounting.

In [9]:
df["price_discount_interaction"] = df["price"] * df["discount"]

## 4.7 Profit Pressure Feature

Represents financial impact intensity of each order.

In [10]:
df["profit_pressure"] = df["total_amount"] * df["profit_margin"]

## 4.8 Encoding Categorical Features

Convert categorical variables into numerical format for ML models.

In [11]:
df = pd.get_dummies(df, columns=["delivery_speed", "discount_band"], drop_first=True)

## 4.9 Remove Leakage Columns

Remove features that are not available at prediction time.

In [12]:
df = df.drop(columns=[
    "return_reason",   # leakage (post-event info)
    "return_date",     # leakage (post-event timestamp)
    "order_id",        # identifier
    "customer_id",     # identifier (unless doing customer modeling)
    "product_id"       # optional identifier
], errors="ignore")